# Family conversation between Husband, wife and brother-in-law

##Tried to optimise token usgae by controlling the conversation state recurringly using openai-oss-120b, used tiktoken to calculate the token and compared within limits to reduce the conversation memory by summarizing

It's just a trail and error. Please free to review, suggest and correct 

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

In [ ]:
groq_url = "https://api.groq.com/openai/v1"
openai = OpenAI(api_key=openai_api_key, base_url=groq_url)

In [ ]:
conversation = """
  husband: Brother we are very supportive to you but we should consider about your wants
  brother: I need it because it'll be nice. Please get me
  wife: Dear we need to think about it...! 
"""
wife_system = """You are a wife and mother of two, who is very kind and lovely sometimes fierce; \
you are a good women, who's supportive to your husband to overcome the situations.Example response should be wife: Hello dear\n"""


In [ ]:
husband_system = """ You are a husband and father of two, who is warm, very responsible and manly; \
who with compassion and lovely to approach anything, 
who's supportive to wife.Example response should be Husband: yes honey\n
"""

brother_system = """ You are a Younger brother of the husband who is active, playful and college student; 
who is living under his brothers(Husband) care.
Example response should be brother: I need a bike brother\n
"""

In [ ]:
def call_husband(conversation):
    user_prompt = f"""Your are husband, 
    who try to make understand you brother an yound adult
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as husband. 
    Example response should be husband: ok let me tell you!
    """
    messages = [{"role": "system", "content": husband_system},{"role": "user", "content": user_prompt}]
    response = openai.chat.completions.create(model="openai/gpt-oss-safeguard-20b", messages=messages)
    return response.choices[0].message.content
    

In [ ]:
def call_brother(conversation):
    user_prompt = f""" Your are brother 
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as brother. 
    Example response should be brother: I like it very much
    """
    messages = [{"role": "system", "content": brother_system},{"role": "user", "content": user_prompt}]
    response = openai.chat.completions.create(model="openai/gpt-oss-safeguard-20b", messages=messages)
    return response.choices[0].message.content

In [ ]:
def call_wife(conversation):
    user_prompt = f"""You are wife, in conversation with husband and brother-in-law.
        The conversation so far is as follows:
        {conversation}
        Now with this, respond with what you would like to say next, as Wife. 
        Example response should be house_wife: please calm down lets talk quietly"""

    messages = [{"role": "system", "content": wife_system},{"role":"user","content":user_prompt}]
    response = openai.chat.completions.create(model="openai/gpt-oss-safeguard-20b", messages=messages)
    return response.choices[0].message.content

In [ ]:
summary_system = f"""You are a summarizer who gothrough the conversation and summarize to maintain the token input limit"""

async def summary_call(conversation:list,input_token:int,limit:int):
  if input_token > limit:
    user_prompt = f"""You are summarizer, who converts the conversation with husband, wife and brother
    into the effective compact summary within token_limit:{limit/2}. Just left the last set of conversation of husband,wife and brother
        The conversation so far is as follows:
        {conversation}
        You should respond in a way of Example:
          They were discussing about the need of bike for the brother
          husband: text
          brother: text
          wife: text
        """
    messages = [{"role":"system","content":summary_system}, {"role":"user","content":user_prompt}]
    response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
    return response.choices[0].message.content
  else: return conversation

In [ ]:
import tiktoken

token_limit = 10000
total_token_sent = 0
summary = ""
async def summarizer(conversation):
    # encoding = tiktoken.encoding_for_model(claude_model)
    # FIX: Explicitly fetch the o200k_base encoding used by the gpt-oss family
    try:
        encoding = tiktoken.get_encoding("o200k_base")
    except ValueError:
        # Fallback to cl100k_base if running a severely outdated tiktoken version
        encoding = tiktoken.get_encoding("cl100k_base")
    cleansed_input = [i.strip() for i in conversation.split("\n")  if i!='']
    for i, request_text in enumerate(cleansed_input):
        # Calculate input tokens for the text
        input_tokens = len(encoding.encode(request_text))
        total_token_sent =+ input_tokens
        if total_token_sent > token_limit:
         summary = await summary_call(conversation,total_token_sent,token_limit)
        else: summary = conversation
    return summary

In [ ]:
for i in range(5):
        
        husband_reply =  call_husband(conversation)
        conversation += f"{husband_reply}\n"
        display(Markdown(f"{husband_reply}\n"))

        brother_reply =  call_brother(conversation)
        conversation += f"{brother_reply}\n"
        display(Markdown(f"{brother_reply}\n"))
        
        wife_reply =  call_wife(conversation)
        conversation += f"{wife_reply}\n"
        display(Markdown(f"{wife_reply}\n"))

        conversation = await summarizer(conversation) 
    